In [ ]:
import os
import re
import warnings

import google.generativeai as genai

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer


warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 140)

RANDOM_STATE = 42
DATA_PATH = "customer_support_tickets_200k.csv"
MAX_ROWS = 200000



In [20]:
raw_df = pd.read_csv(DATA_PATH, nrows=MAX_ROWS)
print(raw_df.shape)
raw_df.head()


(200000, 30)


,ticket_id,customer_name,customer_email,product,category,issue_description,resolution_notes,priority,status,channel,...,ticket_resolved_date,escalated,sla_breached,operating_system,browser,payment_method,language,preferred_contact_time,issue_complexity_score,customer_segment
0,1,Patricia Smith,patricia.smith760@outlook.com,Web Portal,Account Suspension,The payment was deducted from my bank account but the transaction shows failed.,Data synchronization restored after backend service restart.,Urgent,Open,Email,...,2023-05-20,No,Yes,MacOS,Edge,PayPal,French,Afternoon,4,Small Business
1,2,Patricia Williams,patricia.williams390@gmail.com,Mobile App,Performance Issue,I found a bug in the latest update affecting report generation.,Provided step-by-step troubleshooting instructions and issue resolved.,Urgent,Closed,Email,...,2024-01-19,Yes,Yes,Windows,Firefox,PayPal,English,Afternoon,2,Small Business
2,3,William Anderson,william.anderson651@outlook.com,Web Portal,Performance Issue,The application crashes whenever I try to upload a file.,Provided step-by-step troubleshooting instructions and issue resolved.,Medium,Closed,Chat,...,2022-12-05,Yes,Yes,Windows,Safari,Bank Transfer,French,Morning,4,Corporate
3,4,David Miller,david.miller672@icloud.com,Payment Gateway,Subscription Cancellation,My subscription was cancelled without my request and I need clarification.,Provided step-by-step troubleshooting instructions and issue resolved.,Medium,Closed,Social Media,...,2024-04-04,Yes,No,Windows,Chrome,Credit Card,Spanish,Afternoon,7,Corporate
4,5,Robert Gonzalez,robert.gonzalez391@hotmail.com,Web Portal,Feature Request,The system is not syncing data across devices properly.,We have reset the account credentials and advised the customer to retry login.,High,Pending Customer,Email,...,2024-08-24,Yes,No,Linux,NaN,Debit Card,Spanish,Evening,3,Corporate


In [21]:
column_map = {
    "ticket_id": "Ticket ID",
    "category": "Ticket Type",
    "issue_description": "Ticket Description",
    "resolution_notes": "Resolution",
    "priority": "Ticket Priority",
    "customer_satisfaction_score": "Customer Satisfaction Rating",
    "ticket_created_date": "Ticket Created Date",
    "ticket_resolved_date": "Ticket Resolved Date",
    "status": "Ticket Status",
    "product": "Product Purchased"
}
work = raw_df.rename(columns=column_map).copy()

work["Ticket Subject"] = (
    work["Ticket Type"].fillna("Support issue").astype(str)
    + " - "
    + work["Ticket Description"].fillna("").astype(str).str.slice(0, 80)
)

input_columns = [
    "Ticket Subject",
    "Ticket Description",
    "Resolution",
    "Ticket Type",
    "Ticket Priority",
    "Customer Satisfaction Rating"
]

work = work[input_columns + [
    "Ticket ID",
    "Ticket Created Date",
    "Ticket Resolved Date",
    "Ticket Status",
    "Product Purchased",
    "resolution_time_hours",
    "escalated",
    "sla_breached"
]].copy()

work["Customer Satisfaction Rating"] = pd.to_numeric(
    work["Customer Satisfaction Rating"],
    errors="coerce"
)
work["resolution_time_hours"] = pd.to_numeric(
    work["resolution_time_hours"],
    errors="coerce"
)

for column in ["escalated", "sla_breached"]:
    work[column] = (
        work[column]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"yes": 1, "no": 0, "true": 1, "false": 0, "1": 1, "0": 0})
        .fillna(0)
        .astype(int)
    )

print(work.shape)
work[input_columns + ["resolution_time_hours", "escalated", "sla_breached"]].head()


(200000, 14)


,Ticket Subject,Ticket Description,Resolution,Ticket Type,Ticket Priority,Customer Satisfaction Rating,resolution_time_hours,escalated,sla_breached
0,Account Suspension - The payment was deducted from my bank account but the transaction shows failed.,The payment was deducted from my bank account but the transaction shows failed.,Data synchronization restored after backend service restart.,Account Suspension,Urgent,5,108.36,0,1
1,Performance Issue - I found a bug in the latest update affecting report generation.,I found a bug in the latest update affecting report generation.,Provided step-by-step troubleshooting instructions and issue resolved.,Performance Issue,Urgent,5,87.43,1,1
2,Performance Issue - The application crashes whenever I try to upload a file.,The application crashes whenever I try to upload a file.,Provided step-by-step troubleshooting instructions and issue resolved.,Performance Issue,Medium,5,78.50,1,1
3,Subscription Cancellation - My subscription was cancelled without my request and I need clarification.,My subscription was cancelled without my request and I need clarification.,Provided step-by-step troubleshooting instructions and issue resolved.,Subscription Cancellation,Medium,4,239.36,1,0
4,Feature Request - The system is not syncing data across devices properly.,The system is not syncing data across devices properly.,We have reset the account credentials and advised the customer to retry login.,Feature Request,High,5,122.34,1,0


In [22]:
work["ticket_text"] = (
    work["Ticket Subject"].fillna("").astype(str)
    + " "
    + work["Ticket Description"].fillna("").astype(str)
)

work[["Ticket Subject", "Ticket Description", "ticket_text"]].head()


,Ticket Subject,Ticket Description,ticket_text
0,Account Suspension - The payment was deducted from my bank account but the transaction shows failed.,The payment was deducted from my bank account but the transaction shows failed.,Account Suspension - The payment was deducted from my bank account but the transaction shows failed. The payment was deducted from my ba...
1,Performance Issue - I found a bug in the latest update affecting report generation.,I found a bug in the latest update affecting report generation.,Performance Issue - I found a bug in the latest update affecting report generation. I found a bug in the latest update affecting report ...
2,Performance Issue - The application crashes whenever I try to upload a file.,The application crashes whenever I try to upload a file.,Performance Issue - The application crashes whenever I try to upload a file. The application crashes whenever I try to upload a file.
3,Subscription Cancellation - My subscription was cancelled without my request and I need clarification.,My subscription was cancelled without my request and I need clarification.,Subscription Cancellation - My subscription was cancelled without my request and I need clarification. My subscription was cancelled wit...
4,Feature Request - The system is not syncing data across devices properly.,The system is not syncing data across devices properly.,Feature Request - The system is not syncing data across devices properly. The system is not syncing data across devices properly.


In [23]:
import nltk
from nltk.stem import WordNetLemmatizer

# nltk.download("wordnet") run once to download this
# nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(tokens)

work["clean_ticket_text"] = work["ticket_text"].apply(clean_text)
work[["ticket_text", "clean_ticket_text"]].head()



,ticket_text,clean_ticket_text
0,Account Suspension - The payment was deducted from my bank account but the transaction shows failed. The payment was deducted from my ba...,account suspension the payment wa deducted from my bank account but the transaction show failed the payment wa deducted from my bank acc...
1,Performance Issue - I found a bug in the latest update affecting report generation. I found a bug in the latest update affecting report ...,performance issue i found a bug in the latest update affecting report generation i found a bug in the latest update affecting report gen...
2,Performance Issue - The application crashes whenever I try to upload a file. The application crashes whenever I try to upload a file.,performance issue the application crash whenever i try to upload a file the application crash whenever i try to upload a file
3,Subscription Cancellation - My subscription was cancelled without my request and I need clarification. My subscription was cancelled wit...,subscription cancellation my subscription wa cancelled without my request and i need clarification my subscription wa cancelled without ...
4,Feature Request - The system is not syncing data across devices properly. The system is not syncing data across devices properly.,feature request the system is not syncing data across device properly the system is not syncing data across device properly


In [24]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

texts = work["clean_ticket_text"].tolist()

ticket_embeddings = embedding_model.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True
)

ticket_embeddings = np.asarray(ticket_embeddings, dtype="float32")
print(ticket_embeddings.shape)


Batches:   0%|          | 0/782 [00:00<?, ?it/s]

(200000, 384)


In [25]:
import faiss

embedding_dim = ticket_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(ticket_embeddings)

print("Vectors stored:", faiss_index.ntotal)
print("Embedding dimensions:", embedding_dim)


Vectors stored: 200000
Embedding dimensions: 384


In [26]:
def embed_new_ticket(subject, description):
    text = clean_text(str(subject) + " " + str(description))
    embedding = embedding_model.encode(
        [text],
        normalize_embeddings=True
    )
    return np.asarray(embedding, dtype="float32")

def find_similar_tickets(subject, description, top_k=5):
    query_embedding = embed_new_ticket(subject, description)
    scores, indices = faiss_index.search(query_embedding, top_k)

    results = work.iloc[indices[0]].copy()
    results["similarity_score"] = scores[0]
    return results[[
        "Ticket ID",
        "Ticket Subject",
        "Ticket Type",
        "Ticket Priority",
        "Resolution",
        "similarity_score"
    ]]

find_similar_tickets(
    "refund not received",
    "customer says money never came back after refund was approved",
    top_k=5
)


,Ticket ID,Ticket Subject,Ticket Type,Ticket Priority,Resolution,similarity_score
393,394,Refund Request - The payment was deducted from my bank account but the transaction shows failed.,Refund Request,Urgent,Provided step-by-step troubleshooting instructions and issue resolved.,0.644582
390,391,Refund Request - The payment was deducted from my bank account but the transaction shows failed.,Refund Request,Urgent,Subscription status corrected and confirmation email sent to customer.,0.644582
374,375,Refund Request - The payment was deducted from my bank account but the transaction shows failed.,Refund Request,Low,Payment gateway timeout issue fixed and monitoring implemented.,0.644582
137,138,Refund Request - The payment was deducted from my bank account but the transaction shows failed.,Refund Request,Urgent,Data synchronization restored after backend service restart.,0.644582
15,16,Refund Request - The payment was deducted from my bank account but the transaction shows failed.,Refund Request,High,Security settings updated and customer notified of precautionary measures.,0.644582


In [40]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
USE_GEMINI_LLM = os.getenv("USE_GEMINI_LLM", "true").lower() == "true"


def call_gemini_llm(prompt, max_output_tokens=250):
    if not USE_GEMINI_LLM:
        return "Gemini LLM calls are disabled. Set USE_GEMINI_LLM=true to enable live LLM generation."

    if not GEMINI_API_KEY:
        return "GEMINI_API_KEY is not set. Add your Gemini key to the environment and rerun this cell."

    try:
        genai.configure(api_key=GEMINI_API_KEY)
        model = genai.GenerativeModel(GEMINI_MODEL)

        response = model.generate_content(
            prompt,
            generation_config={
                "temperature": 0.2,
                "max_output_tokens": max_output_tokens
            },
            request_options={"timeout": 30}
        )

        return response.text.strip()
    except Exception as exc:
        return f"Gemini LLM unavailable ({type(exc).__name__}): {exc}"


In [41]:
def recommend_response(subject, description, top_k=5):
    similar = find_similar_tickets(subject, description, top_k=top_k)
    resolved = similar[
        similar["Resolution"].notna()
        & (similar["Resolution"].astype(str).str.len() > 0)
    ]

    resolution_examples = "\n".join(
        f"- {text}"
        for text in resolved["Resolution"].astype(str).head(3)
    )

    prompt = f"""
You are a customer support assistant.

New ticket subject: {subject}
New ticket description: {description}

Similar past resolutions:
{resolution_examples}

Write a concise, helpful suggested response for the support agent.
"""

    suggestion = call_gemini_llm(prompt)

    return {
        "suggested_response": suggestion,
        "similar_tickets": similar
    }

example_response = recommend_response(
    "refund not received",
    "customer says money never came back after refund was approved",
    top_k=5
)

print(example_response["suggested_response"])
# example_response["similar_tickets"]



Here's a suggested response for the support agent


In [42]:
N_CLUSTERS = 12

cluster_model = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=10
)

work["issue_cluster"] = cluster_model.fit_predict(ticket_embeddings)

vectorizer = CountVectorizer(max_features=3000, stop_words="english", ngram_range=(1, 2))
term_matrix = vectorizer.fit_transform(work["clean_ticket_text"])
terms = np.array(vectorizer.get_feature_names_out())


def top_terms_for_cluster(cluster_id, top_n=8):
    mask = work["issue_cluster"].values == cluster_id
    mean_terms = np.asarray(term_matrix[mask].mean(axis=0)).ravel()
    top_indices = mean_terms.argsort()[::-1][:top_n]
    return ", ".join(terms[top_indices])

cluster_summary = (
    work.groupby("issue_cluster")
    .agg(
        ticket_count=("Ticket ID", "count"),
        top_ticket_type=("Ticket Type", lambda x: x.mode().iat[0]),
        avg_satisfaction=("Customer Satisfaction Rating", "mean"),
        avg_resolution_hours=("resolution_time_hours", "mean"),
        escalation_rate=("escalated", "mean"),
        sla_breach_rate=("sla_breached", "mean")
    )
    .reset_index()
)

cluster_summary["top_terms"] = cluster_summary["issue_cluster"].apply(top_terms_for_cluster)
cluster_summary.sort_values("ticket_count", ascending=False).head(1)



,issue_cluster,ticket_count,top_ticket_type,avg_satisfaction,avg_resolution_hours,escalation_rate,sla_breach_rate,top_terms
4,4,20232,Subscription Cancellation,3.002867,120.475418,0.506623,0.495898,"statement, billing, discrepancy, discrepancy billing, statement month, month, billing statement, month discrepancy"


In [43]:
def llm_sentiment_for_ticket(ticket_text, rating, priority):
    prompt = f"""
Classify this support ticket into exactly one sentiment label: negative, neutral, or positive.
Also give one short reason.

Ticket: {ticket_text}
Customer satisfaction rating: {rating}
Priority: {priority}

Return format:
label: <negative|neutral|positive>
reason: <short reason>
"""
    return call_gemini_llm(prompt, max_output_tokens=80)

work["priority_score"] = pd.to_numeric(
    work["Ticket Priority"].map({"Low": 1, "Medium": 2, "High": 3, "Critical": 4}),
    errors="coerce"
).fillna(2)

work["frustration_score"] = (
    (5 - work["Customer Satisfaction Rating"].fillna(3)) * 2
    + work["priority_score"]
    + work["escalated"].astype(int) * 2
    + work["sla_breached"].astype(int) * 2
)

work["frustration_level"] = pd.qcut(
    work["frustration_score"].rank(method="first"),
    q=3,
    labels=["low", "medium", "high"]
)

sample_for_llm = work.head(5).copy()
if USE_GEMINI_LLM:
    sample_for_llm["llm_sentiment_analysis"] = sample_for_llm.apply(
        lambda row: llm_sentiment_for_ticket(
            row["clean_ticket_text"],
            row["Customer Satisfaction Rating"],
            row["Ticket Priority"]
        ),
        axis=1
    )
else:
    sample_for_llm["llm_sentiment_analysis"] = "Gemini disabled; using rating-based sentiment proxy downstream."

sample_for_llm[[
    "Ticket ID",
    "Ticket Type",
    "Ticket Priority",
    "Customer Satisfaction Rating",
    "frustration_score",
    "frustration_level",
    "llm_sentiment_analysis"
]]


,Ticket ID,Ticket Type,Ticket Priority,Customer Satisfaction Rating,frustration_score,frustration_level,llm_sentiment_analysis
0,1,Account Suspension,Urgent,5,4.0,low,label: negative
1,2,Performance Issue,Urgent,5,6.0,low,label: negative
2,3,Performance Issue,Medium,5,6.0,low,label:
3,4,Subscription Cancellation,Medium,4,6.0,low,label: negative
4,5,Feature Request,High,5,5.0,low,label


In [46]:
work["Ticket Created Date"] = pd.to_datetime(work["Ticket Created Date"], errors="coerce")
work["month"] = work["Ticket Created Date"].dt.to_period("M").astype(str)

most_common_issues = cluster_summary.sort_values("ticket_count", ascending=False)

high_frustration_categories = (
    work.groupby("Ticket Type")
    .agg(
        tickets=("Ticket ID", "count"),
        avg_frustration=("frustration_score", "mean"),
        high_frustration_rate=("frustration_level", lambda x: (x == "high").mean()),
        avg_satisfaction=("Customer Satisfaction Rating", "mean")
    )
    .reset_index()
    .sort_values("avg_frustration", ascending=False)
)

slowest_resolutions = (
    work.groupby("Ticket Type")
    .agg(
        tickets=("Ticket ID", "count"),
        avg_resolution_hours=("resolution_time_hours", "mean"),
        sla_breach_rate=("sla_breached", "mean")
    )
    .reset_index()
    .sort_values("avg_resolution_hours", ascending=False)
)

monthly_cluster_counts = (
    work.groupby(["month", "issue_cluster"])
    .size()
    .reset_index(name="tickets")
    .sort_values(["issue_cluster", "month"])
)
monthly_cluster_counts["previous_month_tickets"] = monthly_cluster_counts.groupby("issue_cluster")["tickets"].shift(1)
monthly_cluster_counts["mom_change"] = monthly_cluster_counts["tickets"] - monthly_cluster_counts["previous_month_tickets"]

latest_month = monthly_cluster_counts["month"].max()
increasing_complaint_clusters = (
    monthly_cluster_counts[monthly_cluster_counts["month"] == latest_month]
    .merge(cluster_summary, on="issue_cluster", how="left")
    .sort_values("mom_change", ascending=False)
)

insight_context = f"""
Most common issue clusters:
{most_common_issues.head(5).to_string(index=False)}

Increasing complaint clusters:
{increasing_complaint_clusters.head(5).to_string(index=False)}

High-frustration categories:
{high_frustration_categories.head(5).to_string(index=False)}

Slowest resolution categories:
{slowest_resolutions.head(5).to_string(index=False)}
"""

business_insight_prompt = f"""
You are a support operations analyst.
Use the tables below to write 5 concise business insights.
For each insight include the business value or action.

{insight_context}
"""

if USE_GEMINI_LLM:
    llm_business_insights = call_gemini_llm(business_insight_prompt, max_output_tokens=250)
else:
    llm_business_insights = "Gemini disabled; showing computed insight tables below. Set USE_GEMINI_LLM=true to generate narrative insights."

print(llm_business_insights)

print("Most common issues")
display(most_common_issues.head(10))

print("Increasing complaint clusters")
display(increasing_complaint_clusters.head(10))

print("High-frustration categories")
display(high_frustration_categories.head(10))

print("Slowest resolutions")
display(slowest_resolutions.head(1))


Here are 5 concise business insights with
Most common issues


,issue_cluster,ticket_count,top_ticket_type,avg_satisfaction,avg_resolution_hours,escalation_rate,sla_breach_rate,top_terms
4,4,20232,Subscription Cancellation,3.002867,120.475418,0.506623,0.495898,"statement, billing, discrepancy, discrepancy billing, statement month, month, billing statement, month discrepancy"
5,5,20126,Security Concern,2.998211,119.347370,0.502832,0.501491,"payment, account, wa deducted, bank account, transaction, transaction failed, payment wa, failed"
6,6,19988,Login Issue,2.979588,120.955456,0.496748,0.499700,"authentication code, delivered, code, delivered phone, code delivered, factor authentication, factor, phone"
1,1,19978,Feature Request,3.000250,120.564933,0.494494,0.497948,"request, refund, like request, charge, like, recent, recent charge, refund recent"
7,7,19975,Bug Report,3.025832,120.884397,0.503279,0.496521,"performance, experiencing slow, using dashboard, using, dashboard, performance using, slow, experiencing"
9,9,19942,Bug Report,3.008124,121.280016,0.505165,0.504112,"data, device properly, device, data device, syncing, syncing data, properly, properly syncing"
8,8,19912,Subscription Cancellation,2.995781,121.191698,0.500301,0.497640,"request, subscription, cancelled request, subscription wa, wa, request need, need, wa cancelled"
3,3,19907,Refund Request,2.996333,120.339392,0.508515,0.507058,"report, bug, generation, report generation, update affecting, update, affecting, affecting report"
2,2,19883,Feature Request,3.002012,120.307434,0.499472,0.501836,"crash try, try upload, crash, application crash, upload file, upload, application, file"
11,11,13976,Security Concern,3.006583,120.127939,0.504937,0.501646,"access, credential, access account, account, account entering, correct credential, correct, entering"


Increasing complaint clusters


,month,issue_cluster,tickets,previous_month_tickets,mom_change,ticket_count,top_ticket_type,avg_satisfaction,avg_resolution_hours,escalation_rate,sla_breach_rate,top_terms
6,2024-12,6,595,531.0,64.0,19988,Login Issue,2.979588,120.955456,0.496748,0.499700,"authentication code, delivered, code, delivered phone, code delivered, factor authentication, factor, phone"
11,2024-12,11,439,378.0,61.0,13976,Security Concern,3.006583,120.127939,0.504937,0.501646,"access, credential, access account, account, account entering, correct credential, correct, entering"
4,2024-12,4,602,547.0,55.0,20232,Subscription Cancellation,3.002867,120.475418,0.506623,0.495898,"statement, billing, discrepancy, discrepancy billing, statement month, month, billing statement, month discrepancy"
2,2024-12,2,568,518.0,50.0,19883,Feature Request,3.002012,120.307434,0.499472,0.501836,"crash try, try upload, crash, application crash, upload file, upload, application, file"
8,2024-12,8,580,531.0,49.0,19912,Subscription Cancellation,2.995781,121.191698,0.500301,0.497640,"request, subscription, cancelled request, subscription wa, wa, request need, need, wa cancelled"
9,2024-12,9,597,553.0,44.0,19942,Bug Report,3.008124,121.280016,0.505165,0.504112,"data, device properly, device, data device, syncing, syncing data, properly, properly syncing"
5,2024-12,5,557,540.0,17.0,20126,Security Concern,2.998211,119.347370,0.502832,0.501491,"payment, account, wa deducted, bank account, transaction, transaction failed, payment wa, failed"
7,2024-12,7,569,552.0,17.0,19975,Bug Report,3.025832,120.884397,0.503279,0.496521,"performance, experiencing slow, using dashboard, using, dashboard, performance using, slow, experiencing"
0,2024-12,0,59,50.0,9.0,2009,Data Sync Issue,3.031857,118.499791,0.500747,0.514186,"access, access account, account, account entering, correct, entering correct, entering, correct credential"
1,2024-12,1,535,527.0,8.0,19978,Feature Request,3.000250,120.564933,0.494494,0.497948,"request, refund, like request, charge, like, recent, recent charge, refund recent"


High-frustration categories


,Ticket Type,tickets,avg_frustration,high_frustration_rate,avg_satisfaction
8,Security Concern,20040,8.024102,0.337076,2.993513
4,Login Issue,20002,8.022848,0.336916,2.999000
6,Performance Issue,20074,8.021122,0.332769,2.995467
0,Account Suspension,19864,8.021043,0.332511,2.987213
1,Bug Report,19981,8.018267,0.335168,3.000500
3,Feature Request,20169,8.005999,0.337399,2.996034
5,Payment Problem,19997,7.996299,0.333250,3.007701
7,Refund Request,19900,7.994020,0.330955,3.010704
2,Data Sync Issue,19877,7.988077,0.328219,3.008502
9,Subscription Cancellation,20096,7.957355,0.329021,3.012042


Slowest resolutions


,Ticket Type,tickets,avg_resolution_hours,sla_breach_rate
2,Data Sync Issue,19877,121.021393,0.502339


In [45]:
ticket_predictions = work[[
    "Ticket ID",
    "Ticket Subject",
    "Ticket Type",
    "Ticket Priority",
    "Customer Satisfaction Rating",
    "frustration_level",
    "issue_cluster"
]]

ticket_predictions.to_csv("ticket_predictions.csv", index=False)
cluster_summary.to_csv("recurring_issue_clusters.csv", index=False)
high_frustration_categories.to_csv("high_frustration_categories.csv", index=False)
slowest_resolutions.to_csv("slowest_resolutions.csv", index=False)
increasing_complaint_clusters.to_csv("increasing_complaint_clusters.csv", index=False)

print("Saved outputs successfully.")


Saved outputs successfully.


In [ ]:
# Model evaluation metrics and dashboard plots
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    adjusted_rand_score,
    classification_report,
    normalized_mutual_info_score,
    silhouette_score,
)

plt.style.use("seaborn-v0_8-whitegrid")

# Keep evaluation fast and reproducible for the full 200k-row dataset.
RNG = np.random.default_rng(RANDOM_STATE)
eval_sample_size = min(5000, len(work))
eval_indices = RNG.choice(len(work), size=eval_sample_size, replace=False)

pipeline_metrics = []

# 1) Cluster separation quality from the semantic embeddings.
if len(np.unique(work["issue_cluster"])) > 1 and eval_sample_size > len(np.unique(work["issue_cluster"])):
    sampled_embeddings = ticket_embeddings[eval_indices]
    sampled_clusters = work["issue_cluster"].iloc[eval_indices]
    silhouette = silhouette_score(sampled_embeddings, sampled_clusters)
else:
    silhouette = np.nan

pipeline_metrics.append({
    "component": "Issue clustering",
    "metric": "Silhouette score",
    "value": silhouette,
    "interpretation": "Higher is better; measures separation of embedding clusters."
})

# 2) Cluster alignment with the known ticket type label.
cluster_type_counts = pd.crosstab(work["issue_cluster"], work["Ticket Type"])
cluster_purity = cluster_type_counts.max(axis=1).sum() / cluster_type_counts.to_numpy().sum()
cluster_nmi = normalized_mutual_info_score(work["Ticket Type"], work["issue_cluster"])
cluster_ari = adjusted_rand_score(work["Ticket Type"], work["issue_cluster"])

pipeline_metrics.extend([
    {
        "component": "Issue clustering",
        "metric": "Cluster purity vs ticket type",
        "value": cluster_purity,
        "interpretation": "Share of tickets matching the majority ticket type inside their cluster."
    },
    {
        "component": "Issue clustering",
        "metric": "Normalized mutual information",
        "value": cluster_nmi,
        "interpretation": "0 means no label agreement; 1 means perfect agreement."
    },
    {
        "component": "Issue clustering",
        "metric": "Adjusted Rand index",
        "value": cluster_ari,
        "interpretation": "Chance-adjusted agreement between clusters and ticket type labels."
    },
])

# 3) Frustration score sanity check against a transparent business-rule proxy.
frustration_proxy = np.where(
    work["Customer Satisfaction Rating"].le(2) | work["sla_breached"].astype(bool) | work["escalated"].astype(bool),
    "high",
    np.where(work["Customer Satisfaction Rating"].ge(4), "low", "medium")
)
frustration_report = pd.DataFrame(
    classification_report(
        frustration_proxy,
        work["frustration_level"].astype(str),
        labels=["low", "medium", "high"],
        output_dict=True,
        zero_division=0,
    )
).T

pipeline_metrics.append({
    "component": "Frustration scoring",
    "metric": "Macro F1 vs business-rule proxy",
    "value": frustration_report.loc["macro avg", "f1-score"],
    "interpretation": "Sanity-check agreement with a simple rating/SLA/escalation rule."
})

pipeline_evaluation = pd.DataFrame(pipeline_metrics)
pipeline_evaluation["value"] = pipeline_evaluation["value"].astype(float)
pipeline_evaluation.to_csv("pipeline_evaluation.csv", index=False)

print("Pipeline evaluation metrics")
display(pipeline_evaluation)

print("Frustration classification report vs proxy")
display(frustration_report)

# Plot 1: ticket count by issue cluster.
cluster_plot = cluster_summary.sort_values("ticket_count", ascending=False).head(12).copy()
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(cluster_plot["issue_cluster"].astype(str), cluster_plot["ticket_count"], color="#1f6feb")
ax.set_title("Top Issue Clusters by Ticket Count")
ax.set_xlabel("Issue cluster")
ax.set_ylabel("Tickets")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

# Plot 2: satisfaction and SLA risk by cluster.
fig, ax1 = plt.subplots(figsize=(11, 5))
ordered = cluster_summary.sort_values("ticket_count", ascending=False).head(12).copy()
x = np.arange(len(ordered))
ax1.bar(x - 0.18, ordered["avg_satisfaction"], width=0.36, color="#2ca25f", label="Avg satisfaction")
ax1.set_ylabel("Average satisfaction")
ax1.set_ylim(0, 5)
ax2 = ax1.twinx()
ax2.bar(x + 0.18, ordered["sla_breach_rate"], width=0.36, color="#f59f00", label="SLA breach rate")
ax2.set_ylabel("SLA breach rate")
ax2.set_ylim(0, max(0.1, ordered["sla_breach_rate"].max() * 1.25))
ax1.set_xticks(x)
ax1.set_xticklabels(ordered["issue_cluster"].astype(str))
ax1.set_title("Cluster Satisfaction and SLA Risk")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
plt.tight_layout()
plt.show()

# Plot 3: frustration level distribution.
frustration_counts = work["frustration_level"].astype(str).value_counts().reindex(["low", "medium", "high"])
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(frustration_counts.index, frustration_counts.values, color=["#2ca25f", "#f59f00", "#d94841"])
ax.set_title("Frustration Level Distribution")
ax.set_xlabel("Frustration level")
ax.set_ylabel("Tickets")
plt.tight_layout()
plt.show()

# Plot 4: highest-frustration ticket types.
frustration_by_type = (
    work.assign(high_frustration=work["frustration_level"].astype(str).eq("high"))
    .groupby("Ticket Type")
    .agg(
        tickets=("Ticket ID", "count"),
        high_frustration_rate=("high_frustration", "mean"),
        avg_satisfaction=("Customer Satisfaction Rating", "mean"),
    )
    .query("tickets >= 25")
    .sort_values("high_frustration_rate", ascending=False)
    .head(10)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(frustration_by_type["Ticket Type"], frustration_by_type["high_frustration_rate"], color="#d94841")
ax.invert_yaxis()
ax.set_title("Ticket Types with Highest High-Frustration Rate")
ax.set_xlabel("High-frustration rate")
ax.set_ylabel("Ticket type")
plt.tight_layout()
plt.show()

# Plot 5: monthly trend for the largest recurring clusters, if month data exists.
if "monthly_cluster_counts" in globals() and len(monthly_cluster_counts):
    top_cluster_ids = cluster_summary.sort_values("ticket_count", ascending=False).head(5)["issue_cluster"]
    trend_data = monthly_cluster_counts[monthly_cluster_counts["issue_cluster"].isin(top_cluster_ids)].copy()
    trend_pivot = trend_data.pivot_table(
        index="month",
        columns="issue_cluster",
        values="tickets",
        aggfunc="sum",
        fill_value=0,
    ).sort_index()

    fig, ax = plt.subplots(figsize=(12, 5))
    for cluster_id in trend_pivot.columns:
        ax.plot(trend_pivot.index, trend_pivot[cluster_id], marker="o", linewidth=2, label=f"Cluster {cluster_id}")
    ax.set_title("Monthly Ticket Trend for Top Issue Clusters")
    ax.set_xlabel("Month")
    ax.set_ylabel("Tickets")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(ncol=2)
    plt.tight_layout()
    plt.show()
else:
    print("monthly_cluster_counts is not available, so the monthly trend plot was skipped.")
